# taintwall L1 detector — train on Colab (free T4)

Runs [`training/train_l1.py`](train_l1.py), the same script a local or Modal run uses, so
there is one recipe: ModernBERT-base, 2 epochs, lr 2e-5, batch 32, seed 20261020.

What the script does, in order (the order is the guarantee):
1. **Contamination audit** on train/dev/test; it refuses to train if any row overlaps AgentDojo.
2. Fine-tune on **train** only.
3. Pick each model's threshold on **dev** for FPR ≤ 1%, then freeze it.
4. Score the held-out **test** split and **InjecAgent** (MIT, out-of-distribution, eval only)
   for our model and the `protectai/deberta-v3-base-prompt-injection-v2` baseline.

**Before you run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
About 30–45 min on a T4 (fp32; the T4 has no bf16). The last cell downloads
`l1_outputs.zip` (weights + metrics + per-row scores). Weights are never committed to git.

## 1. Environment

In [ ]:
import torch
assert torch.cuda.is_available(), "Set Runtime -> Change runtime type -> T4 GPU"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

## 2. Code and data

Clone the branch and rebuild the dataset from the pinned HF seeds (~1 min). The hashes should match the local build.

In [ ]:
import pathlib
REPO = "https://github.com/YuvGev00/taintwall.git"
BRANCH = "feat/l1-detector"  # main once merged
%cd /content
if not pathlib.Path("taintwall").exists():
    !git clone -q -b $BRANCH $REPO
%cd /content/taintwall
# Always run the latest code on the branch (a re-run otherwise keeps a stale clone).
!git fetch -q origin $BRANCH && git reset -q --hard origin/$BRANCH && git log --oneline -1
!pip -q install -e . "datasets>=5.1.0" "transformers==5.20.2"
if not pathlib.Path("data/train.jsonl").exists():
    !python -m training.build_dataset
!shasum -a 256 data/*.jsonl | cut -c1-16
# expected (local build): eb77b489b9ca85b6 train, 794ac2266ccd10ec dev, 2a47a29187ff976c test

## 3. Audit, train, threshold on dev, evaluate

In [ ]:
!python -m training.train_l1 --device cuda

## 4. Download the outputs

Unzip `l1_outputs.zip` at the repo root: it fills `runs/l1-model/` (gitignored), `runs/l1_scores.json` and `results/l1/metrics.json`.

In [ ]:
!zip -qr l1_outputs.zip runs/l1-model runs/l1_scores.json results/l1
from google.colab import files
files.download("l1_outputs.zip")